In [ ]:
# =========================
# nb_tpir_check
# =========================
# Phase 8, SC-11: Validate tpir_load_equivalent schema and data quality
# Checks for missing ISINs, security_master enrichment, completeness

# ---------- Parameters ----------
period = "2026-03"
run_id = "manual_test_run"

# ---------- Imports ----------
from pyspark.sql import functions as F
import json
import sys

# ---------- Pre-flight checks ----------
try:
    # Verify Fabric notebook context
    mssparkutils
except NameError:
    print("[ERROR] mssparkutils not available - not running in Fabric context")
    sys.exit(1)

try:
    # Verify period/run_id are non-empty
    if not period or not period.strip():
        raise ValueError("period parameter is empty")
    if not run_id or not run_id.strip():
        raise ValueError("run_id parameter is empty")
    print(f"[nb_tpir_check] START period={period}, run_id={run_id}")
except ValueError as e:
    print(f"[ERROR] Invalid parameters: {e}")
    sys.exit(1)

# Verify tables exist
try:
    tpir_table_exists = spark.catalog.tableExists("tpir_load_equivalent")
    result_table_exists = spark.catalog.tableExists("tpir_check_result")
    
    if not tpir_table_exists:
        print("[ERROR] tpir_load_equivalent table does not exist")
        sys.exit(1)
    if not result_table_exists:
        print("[ERROR] tpir_check_result table does not exist")
        sys.exit(1)
except Exception as e:
    print(f"[ERROR] Failed to verify tables: {e}")
    sys.exit(1)

# ---------- Helpers ----------
def load_security_master():
    """Load security_master.csv for ISIN validation."""
    try:
        path = "/lakehouse/default/Files/config/security_master.csv"
        df = spark.read.option("header", "true").option("inferSchema", "true").csv(path)
        print(f"[nb_tpir_check] Loaded security_master.csv: {df.count()} rows")
        return df
    except FileNotFoundError:
        print("[nb_tpir_check] Info: security_master.csv not found (optional)")
        return None
    except Exception as e:
        print(f"[nb_tpir_check] Warning: Failed to load security_master.csv: {type(e).__name__}: {e}")
        return None

# ---------- Load tpir data ----------
try:
    tpir = spark.table("tpir_load_equivalent").filter(
        (F.col("period") == period) & (F.col("run_id") == run_id)
    )
    
    if tpir.rdd.isEmpty():
        print(f"[nb_tpir_check] No tpir_load_equivalent rows for period={period}, run_id={run_id}")
        spark.sql(f"""
            INSERT INTO tpir_check_result
            (period, run_id, dfm_id, status, missing_isins_count, validated_rows, validation_errors, validation_completed_at)
            VALUES
            ('{period}', '{run_id}', 'orchestrator', 'SKIPPED', 0, 0, 'NO_DATA', current_timestamp())
        """)
        mssparkutils.notebook.exit("NO_DATA")
except Exception as e:
    print(f"[ERROR] Failed to load tpir_load_equivalent: {type(e).__name__}: {e}")
    sys.exit(1)

# ---------- Validation checks ----------
try:
    # Check 1: Required columns present
    required_tpir_cols = [
        "period", "run_id", "dfm_id", "policy_id", "isin", "security_name",
        "quantity", "market_value_gbp", "currency", "valuation_date"
    ]
    actual_cols = tpir.columns
    missing_cols = [c for c in required_tpir_cols if c not in actual_cols]
    
    # Check 2: Missing ISINs
    missing_isins = tpir.filter(F.col("isin").isNull() | (F.col("isin") == "")).count()
    total_rows = tpir.count()
    
    # Check 3: Duplicate ISINs per policy (should not exist)
    duplicates = tpir.groupBy("policy_id", "isin").count().filter(F.col("count") > 1).count()
    
    # Check 4: Security master enrichment (if available)
    sm = load_security_master()
    isins_in_master = 0
    if sm is not None:
        try:
            tpir_with_master = tpir.join(
                sm.select(F.col("isin").alias("isin_master")),
                F.col("isin") == F.col("isin_master"),
                "left_semi"
            ).count()
            isins_in_master = tpir_with_master
            print(f"[nb_tpir_check] ISINs in security_master: {isins_in_master}/{total_rows}")
        except Exception as e:
            print(f"[nb_tpir_check] Warning: Security master enrichment failed: {e}")
    
    # Overall status
    status = "passed"
    errors = []
    
    if missing_cols:
        status = "failed"
        errors.append(f"Missing columns: {missing_cols}")
    
    if missing_isins > 0:
        status = "failed" if missing_isins == total_rows else "degraded"
        errors.append(f"Missing ISINs: {missing_isins} rows")
    
    if duplicates > 0:
        status = "failed"
        errors.append(f"Duplicate ISINs per policy: {duplicates} groups")
    
    validation_errors_json = json.dumps({
        "missing_columns": missing_cols,
        "missing_isins_count": missing_isins,
        "duplicate_groups": duplicates,
        "isins_in_security_master": isins_in_master,
        "total_rows": total_rows,
        "errors": errors
    })
    
    # Write result with error handling
    spark.sql(f"""
        INSERT INTO tpir_check_result
        (period, run_id, dfm_id, status, missing_isins_count, validated_rows, validation_errors, validation_completed_at)
        VALUES
        ('{period}', '{run_id}', 'orchestrator', '{status}', {missing_isins}, {total_rows}, '{validation_errors_json.replace("'", "''")}', current_timestamp())
    """)
    
    print(f"[nb_tpir_check] COMPLETE status={status}")
    print(f"  Total rows: {total_rows}")
    print(f"  Missing ISINs: {missing_isins}")
    print(f"  Duplicates: {duplicates}")
    if errors:
        print(f"  Issues: {'; '.join(errors)}")
    
    mssparkutils.notebook.exit(status.upper())

except Exception as e:
    print(f"[ERROR] Validation failed: {type(e).__name__}: {e}")
    import traceback
    traceback.print_exc()
    sys.exit(1)